## Ingest data from API football 

In [0]:
import json
import os
import time
from datetime import date, datetime, timezone

import requests

In [0]:
dbutils.widgets.dropdown("mode", "historical", ["historical", "live"], "Mode")
MODE = dbutils.widgets.get("mode")

RAW_PATH = "/Volumes/football_scouting/default/raw"
BASE_URL = "https://v3.football.api-sports.io"


# Leagues to scout - ordered by priority (loaded from top to bottom).
LEAGUES = {
    # Europe - top 5
    39: "Premier League (England)",
    140: "La Liga (Spain)",
    135: "Serie A (Italy)",
    78: "Bundesliga (Germany)",
    61: "Ligue 1 (France)",
    # Europe - other interesting scouting markets
    88: "Eredivisie (Netherlands)",
    94: "Primeira Liga (Portugal)",
    144: "Jupiler Pro League (Belgium)",
    40: "Championship (England)",
    203: "Super Lig (Turkey)",
    179: "Premiership (Scotland)",
    218: "Bundesliga (Austria)",
    207: "Super League (Switzerland)",
    119: "Superliga (Denmark)",
    113: "Allsvenskan (Sweden)",
    103: "Eliteserien (Norway)",
    # South America
    71: "Serie A (Brazil)",
    128: "Liga Profesional (Argentina)",
    # North America
    253: "Major League Soccer (USA)",
    262: "Liga MX (Mexico)",
}

# Load every season from the year 2022 up to the current one.
FIRST_SEASON = 2022

MAX_CALLS_PER_RUN = 6000
SECONDS_BETWEEN_CALLS = 0.25

print(f"Mode: {MODE} | {len(LEAGUES)} leagues | seasons from {FIRST_SEASON} | max {MAX_CALLS_PER_RUN} calls")




##  API key (secret) - reads the key from the Databricks secret scope `football`

In [0]:
# API key (secret)  reads the key from the Databricks secret scope `football`

try:
    API_KEY = dbutils.secrets.get(scope="football", key="api_football_key")
    print("API key loaded from secret scope 'football'")
except Exception:
    dbutils.widgets.text("api_key", "", "API key (only if no secret)")
    API_KEY = dbutils.widgets.get("api_key")

if not API_KEY:
    raise ValueError("No API key found. Create the secret or paste the key in the 'api_key' box.")

## Calling the API

In [0]:
class BudgetUsedUp(Exception):
    """We used all API calls we allow ourselves in this run (or the daily limit is reached)."""


class SeasonNotInPlan(Exception):
    """Our API plan doesn't include this league/season."""


calls_made = 0

def call_api(endpoint, params=None):
    """Call one API-Football endpoint and return the JSON answer (a dict)."""
    global calls_made

    while True:
        if calls_made >= MAX_CALLS_PER_RUN:
            raise BudgetUsedUp()

        time.sleep(SECONDS_BETWEEN_CALLS)  # stay under the "calls per minute" limit
        response = requests.get(
            f"{BASE_URL}/{endpoint}",
            headers={"x-apisports-key": API_KEY},
            params=params,
            timeout=60,
        )
        calls_made += 1
        response.raise_for_status()
        body = response.json()

        # API-Football reports problems in the "errors" field (the HTTP status is still 200)
        errors = body.get("errors")
        if not errors:
            return body

        message = json.dumps(errors)
        if "rateLimit" in message:
            print("Too many calls per minute - waiting 60 seconds...")
            time.sleep(60)
        elif "plan" in message:
            raise SeasonNotInPlan(message)
        elif "requests" in message:
            print(f"Daily limit reached: {message}")
            raise BudgetUsedUp()
        else:
            raise Exception(f"API error for {endpoint} {params}: {message}")


def call_api_all_pages(endpoint, params):
    """Some endpoints (like players) returns 20 records per page. Get all pages."""
    records = []
    page = 1
    while True:
        body = call_api(endpoint, {**params, "page": page})
        records += body["response"]
        if page >= body["paging"]["total"]:
            return records
        page += 1

## Saving files

In [0]:
"""
    Saves a list of API records as a JSON-lines file:
    /Volumes/football_scouting/default/raw/<entity>/<entity>_<timestamp>.json

    Every line: {"entity", "league_id", "season", "ingested_at", "raw_json": "<record as text>"}
    Added league_id and season because some records (like teams) don't contain them.
    """
def save_records(entity, records, league_id=None, season=None):
    if not records:
        return

    now = datetime.now(timezone.utc)
    file_path = f"{RAW_PATH}/{entity}/{entity}_{now:%Y%m%d_%H%M%S_%f}.json"
    os.makedirs(os.path.dirname(file_path), exist_ok=True)

    with open(file_path, "w") as file:
        for record in records:
            line = {
                "entity": entity,
                "league_id": league_id,
                "season": season,
                "ingested_at": now.isoformat(),
                "raw_json": json.dumps(record),
            }
            file.write(json.dumps(line) + "\n")

    print(f"  saved {len(records)} {entity}")


# Progress: an empty file raw/_progress/<league>_<season>.done means "this finished season is loaded".
def progress_file(league_id, season):
    return f"{RAW_PATH}/_progress/{league_id}_{season}.done"


def is_done(league_id, season):
    return os.path.exists(progress_file(league_id, season))


def mark_done(league_id, season):
    os.makedirs(f"{RAW_PATH}/_progress", exist_ok=True)
    open(progress_file(league_id, season), "w").close()


## Load historical


In [0]:
def load_leagues():
    """
    One call returns ALL leagues with their seasons .
    saves them on every run, so the pipeline always knows about new seasons.
    """
    all_leagues = call_api("leagues")["response"]
    our_leagues = [item for item in all_leagues if item["league"]["id"] in LEAGUES]
    save_records("leagues", our_leagues)
    return our_leagues


def seasons_to_load(leagues):
    """
    Make the to-do list: (league_id, season, is_finished) for every season from FIRST_SEASON on.
    Newest season first, then the leagues in the order of LEAGUES.
    """
    today = date.today().isoformat()  # e.g. "2026-11-03" - dates as text compare correctly
    todo = []

    for item in leagues:
        for season in item["seasons"]:
            if season["year"] < FIRST_SEASON or season["start"] > today:
                continue  # too old, or not started yet
            is_finished = season["end"] < today
            todo.append((item["league"]["id"], season["year"], is_finished))

    league_order = list(LEAGUES)
    todo.sort(key=lambda row: (-row[1], league_order.index(row[0])))
    return todo


def load_league_season(league_id, season):
    """Teams, matches and players (with season statistics) of one league in one season."""
    params = {"league": league_id, "season": season}
    save_records("teams", call_api("teams", params)["response"], league_id, season)
    save_records("fixtures", call_api("fixtures", params)["response"], league_id, season)
    save_records("players", call_api_all_pages("players", params), league_id, season)


def load_historical():
    leagues = load_leagues()

    for league_id, season, is_finished in seasons_to_load(leagues):
        if is_finished and is_done(league_id, season):
            continue  # finished season, already loaded

        status = "finished" if is_finished else "current season - refreshed every run"
        print(f"{LEAGUES[league_id]} - season {season} ({status})")

        try:
            load_league_season(league_id, season)
        except SeasonNotInPlan as error:
            print(f"  skipped - not in your API plan: {error}")
            continue

        if is_finished:
            mark_done(league_id, season)

## Live loading (streaming)

In [0]:
"""One call returns all matches being played right now. Every run creates a new file,
the bronze streaming table only reads the new files."""

def load_live():
    live = call_api("fixtures", {"live": "all"})["response"]
    our_matches = [match for match in live if match["league"]["id"] in LEAGUES]
    
    # league and season are inside every match record, so it is not passed here
    save_records("fixtures", our_matches)
    print(f"{len(our_matches)} live matches in our leagues")

## Final Run 

In [0]:
try:
    if MODE == "historical":
        load_historical()
    else:
        load_live()
    print("Done!")
except BudgetUsedUp:
    print("API budget for this run is used up. The next run continues from here.")

print(f"API calls used in this run: {calls_made}")